# 🤖 Pertemuan 13 — Python ML: Membangun ML App dengan Streamlit
### Mata Kuliah: Python Machine Learning

---

## Cara Menggunakan Notebook Ini

| Langkah | Aksi |
|---------|------|
| **1** | Jalankan sel **Setup** (pertama) untuk membuat folder & cek instalasi |
| **2** | Baca penjelasan di setiap bagian |
| **3** | Jalankan sel `%%writefile` untuk membuat file `.py` demo |
| **4** | Klik link yang muncul untuk menjalankan app di browser |
| **5** | Eksplorasi! Ubah kode & lihat perubahannya langsung |

> **PENTING:** Setiap file `.py` harus dijalankan dengan `streamlit run`. Gunakan tombol ▶️ yang muncul di setiap sel, atau klik link yang ditampilkan.

---

## Tujuan Pembelajaran

Setelah pertemuan ini, kamu bisa:
1. Menjelaskan apa itu Streamlit dan mengapa ia populer untuk prototyping ML app
2. Menggunakan komponen teks, widget input, dan layout Streamlit
3. Menampilkan visualisasi matplotlib/seaborn di dalam Streamlit app
4. Membangun aplikasi EDA interaktif sederhana yang siap dijalankan

---

## Recap Cepat — Quiz Kilat ⚡

Sebelum mulai, jawab pertanyaan berikut tanpa melihat catatan:

1. Dari P12: Apa perbedaan **ARIMA** dan **SARIMA**? Kapan SARIMA lebih cocok?
2. Mengapa train/test split pada data timeseries harus **temporal** (tidak boleh acak)?
3. Apa itu **MAPE** dan mengapa ia lebih mudah dikomunikasikan ke non-teknis dibanding MAE?

---

In [1]:
import os, sys, subprocess, time, threading, webbrowser

os.makedirs('../streamlit_apps/p13_demo', exist_ok=True)

try:
    import streamlit
    print(f"✅ Streamlit {streamlit.__version__} sudah terinstall!")
except ImportError:
    print("❌ Streamlit belum terinstall. Jalankan: pip install streamlit")
    sys.exit(1)

print("\n📁 Folder demo siap: streamlit_apps/p13_demo/\n")

# ── Helper: jalankan app Streamlit dari notebook ──────────
PORTS = iter(range(8501, 8520))

def launch(name):
    """Launch a Streamlit app from streamlit_apps/p13_demo/ in browser."""
    path = os.path.abspath(f'../streamlit_apps/p13_demo/{name}')
    if not os.path.exists(path):
        print(f"❌ File belum ada. Jalankan sel %%writefile dulu: {path}")
        return
    port = next(PORTS)
    def _run():
        subprocess.run([sys.executable, '-m', 'streamlit', 'run', path,
                        '--server.port', str(port), '--server.headless', 'true'])
    threading.Thread(target=_run, daemon=True).start()
    time.sleep(2)
    url = f'http://localhost:{port}'
    webbrowser.open(url)
    print(f'🚀 App berjalan di: {url}')
    print(f'   Tekan Ctrl+C di terminal untuk menghentikan')

print("💡 Cara pakai:")
print("   1. Jalankan sel %%writefile untuk membuat file .py")
print("   2. Klik link ▶️ atau ketik: launch('<nama_file>.py')")

✅ Streamlit 1.56.0 sudah terinstall!

📁 Folder demo siap: streamlit_apps/p13_demo/

💡 Cara pakai:
   1. Jalankan sel %%writefile untuk membuat file .py
   2. Klik link ▶️ atau ketik: launch('<nama_file>.py')


---

## BAGIAN 1: Apa itu Streamlit?

**Streamlit** adalah framework Python open-source yang memungkinkan kamu membangun **web app interaktif** hanya dari script Python biasa — tanpa HTML, CSS, atau JavaScript.

```
Tanpa Streamlit:                        Dengan Streamlit:
──────────────────────────────          ──────────────────────────────
Python model → Flask/Django →           Python model
HTML template → CSS styling →           + 10 baris Streamlit
JavaScript interactivity                = Web app interaktif ✅

Waktu: berminggu-minggu                 Waktu: hitungan jam
```

### Cara Kerja Streamlit

```
Script Python kamu (.py):
  ┌─────────────────────────────────────┐
  │  import streamlit as st             │
  │  st.title("Hello!")                 │
  │  name = st.text_input("Nama kamu?") │
  │  st.write(f"Halo, {name}!")         │
  └─────────────────────────────────────┘
            ↓  streamlit run app.py
  ┌─────────────────────────────────────┐
  │  Browser: localhost:8501            │
  │  ┌───────────────────────────────┐  │
  │  │  Hello!                       │  │
  │  │  Nama kamu? [____________]    │  │
  │  │  Halo, Budi!                  │  │
  │  └───────────────────────────────┘  │
  └─────────────────────────────────────┘
```

**Siklus re-run:** Setiap kali pengguna berinteraksi (slider bergerak, tombol ditekan, input berubah), **seluruh script dijalankan ulang dari atas ke bawah**. Inilah mengapa `@st.cache_data` penting untuk operasi berat (kita pelajari di P14).

### Instalasi

```bash
pip install streamlit
```

Verifikasi:
```bash
streamlit hello   # membuka demo app bawaan
```

### Perbedaan Notebook vs Streamlit

| Aspek | Jupyter Notebook | Streamlit App |
|-------|-----------------|---------------|
| **Format file** | `.ipynb` | `.py` |
| **Cara jalankan** | `jupyter notebook` | `streamlit run app.py` |
| **Pengguna** | Data scientist / developer | End user / stakeholder |
| **Interaktivitas** | Terbatas (widget ipython) | Penuh (slider, input, button) |
| **Deploy** | Sulit | Mudah (Streamlit Cloud, gratis) |
| **Tujuan** | Eksplorasi & analisis | Produk / demo / presentasi |

---
▶️ **Setelah menjalankan sel berikut**, ketik `launch('01_hello.py')` untuk melihat hasilnya di browser.

In [2]:
%%writefile ../streamlit_apps/p13_demo/01_hello.py
# ============================================================
# Demo 01 — Struktur Dasar Streamlit
# Jalankan: streamlit run streamlit_apps/p13_demo/01_hello.py
# ============================================================
import streamlit as st

# Konfigurasi halaman — SELALU taruh di baris pertama setelah import
st.set_page_config(
    page_title="Hello Streamlit!",
    page_icon="👋",
    layout="centered",   # 'centered' atau 'wide'
)

# ── Judul & Teks ────────────────────────────────────────────
st.title("👋 Hello, Streamlit!")
st.header("Ini adalah Header")
st.subheader("Ini adalah Subheader")

st.write("st.write() bisa tampilkan apa saja: teks, angka, DataFrame, chart!")
st.markdown("**Markdown** juga *didukung* — bisa pakai `code inline`, bullet list, dll.")

# ── Input Sederhana ─────────────────────────────────────────
nama = st.text_input("Siapa nama kamu?", placeholder="Masukkan nama...")

if nama:
    st.success(f"Halo, **{nama}**! Selamat datang di Streamlit 🎉")
else:
    st.info("Masukkan namamu di atas untuk mendapat sapaan!")

Overwriting ../streamlit_apps/p13_demo/01_hello.py


---

## BAGIAN 2: Komponen Teks & Display

### Fungsi Teks Utama

| Fungsi | Kegunaan | Contoh |
|--------|---------|--------|
| `st.title()` | Judul halaman (H1) | `st.title("Dashboard Penjualan")` |
| `st.header()` | Judul bagian (H2) | `st.header("Analisis Bulanan")` |
| `st.subheader()` | Sub-judul (H3) | `st.subheader("Januari 2024")` |
| `st.write()` | Teks serbaguna — otomatis deteksi tipe data | `st.write(df)` |
| `st.markdown()` | Teks dengan Markdown syntax | `st.markdown("**Tebal** *miring*")` |
| `st.text()` | Teks plain monospace | `st.text("output raw")` |
| `st.code()` | Blok kode dengan syntax highlight | `st.code("print('hi')", language='python')` |

### Fungsi Display Lainnya

| Fungsi | Kegunaan |
|--------|---------|
| `st.metric()` | Tampilkan KPI/angka dengan delta (naik/turun) |
| `st.success()` | Kotak hijau — pesan sukses |
| `st.error()` | Kotak merah — pesan error |
| `st.warning()` | Kotak kuning — peringatan |
| `st.info()` | Kotak biru — informasi |
| `st.divider()` | Garis pemisah horizontal |
| `st.balloons()` | Animasi balon 🎈 |

### st.metric() — Menampilkan KPI

```python
st.metric(
    label="Total Pendapatan",
    value="Rp 1.2M",
    delta="+12%",        # hijau jika positif, merah jika negatif
    delta_color="normal" # 'normal', 'inverse', 'off'
)
```

---
▶️ **Setelah menjalankan sel berikut**, ketik `launch('02_teks_display.py')`

In [5]:
%%writefile ../streamlit_apps/p13_demo/02_teks_display.py
# ============================================================
# Demo 02 — Komponen Teks & Display
# Jalankan: streamlit run streamlit_apps/p13_demo/02_teks_display.py
# ============================================================
import streamlit as st

st.set_page_config(page_title="Teks & Display", page_icon="📝", layout="wide")

st.title("📝 Komponen Teks & Display")
st.divider()

# ── Hierarki judul ─────────────────────────────────────────
st.header("1. Hierarki Judul")
col1, col2 = st.columns(2)
with col1:
    st.title("st.title()")
    st.header("st.header()")
    st.subheader("st.subheader()")
with col2:
    st.markdown("**st.markdown()** — *italic*, `code`, [link](https://streamlit.io)")
    st.write("st.write() menampilkan: teks, angka, DataFrame, chart, markdown")
    st.text("st.text() — plain monospace\ncocok untuk raw output")
    st.code("x = [i**2 for i in range(5)]\nprint(x)", language="python")

st.divider()

# ── Alert messages ─────────────────────────────────────────
st.header("2. Alert & Status")
col1, col2 = st.columns(2)
with col1:
    st.success("✅ Model berhasil dilatih! Akurasi: 94.2%")
    st.error("❌ File tidak ditemukan. Cek path-nya.")
with col2:
    st.warning("⚠️ Jumlah data terlalu sedikit untuk cross-validation.")
    st.info("ℹ️ Model menggunakan 80% data untuk training.")

st.divider()

# ── Metrics ───────────────────────────────────────────────
st.header("3. Metrics — KPI Dashboard")
col1, col2, col3, col4 = st.columns(4)
col1.metric("Akurasi Model", "94.2%",  "+2.1%")
col2.metric("Jumlah Data",   "10,482", "+1,203")
col3.metric("F1-Score",      "0.93",   "-0.01",  delta_color="inverse")
col4.metric("Waktu Prediksi","12 ms",  "-3 ms",   delta_color="inverse")

Overwriting ../streamlit_apps/p13_demo/02_teks_display.py


---

## BAGIAN 3: Widget Input

Widget adalah komponen interaktif yang **mengembalikan nilai** ke variabel Python. Setiap kali widget berubah, script dijalankan ulang dengan nilai baru.

```
Widget → nilai Python → dipakai di logika app

  slider = st.slider("Pilih K", 1, 10, 3)
  # slider sekarang berisi integer, misal: 3

  if slider > 5:
      st.warning("K terlalu besar!")
```

### Daftar Widget Umum

| Widget | Fungsi | Return Type |
|--------|--------|-------------|
| `st.text_input()` | Input teks satu baris | `str` |
| `st.text_area()` | Input teks multi-baris | `str` |
| `st.number_input()` | Input angka | `int` / `float` |
| `st.slider()` | Slider angka (atau rentang) | `int` / `float` / `tuple` |
| `st.selectbox()` | Dropdown pilih satu | nilai yang dipilih |
| `st.multiselect()` | Dropdown pilih banyak | `list` |
| `st.radio()` | Radio button | nilai yang dipilih |
| `st.checkbox()` | Toggle on/off | `bool` |
| `st.toggle()` | Toggle switch | `bool` |
| `st.button()` | Tombol | `bool` (True sekali saat diklik) |
| `st.file_uploader()` | Upload file | `UploadedFile` / `None` |
| `st.date_input()` | Pilih tanggal | `datetime.date` |
| `st.color_picker()` | Pilih warna | hex string |

### Tips: Widget di Sidebar

```python
# Taruh widget di sidebar agar tidak memenuhi konten utama
with st.sidebar:
    fitur = st.selectbox("Pilih fitur", ["A", "B", "C"])
    n = st.slider("Jumlah data", 10, 1000, 100)

# Atau pakai prefix st.sidebar.*
model = st.sidebar.radio("Model", ["RF", "SVM", "LR"])
```

---
▶️ **Setelah menjalankan sel berikut**, ketik `launch('03_widgets.py')`

In [6]:
%%writefile ../streamlit_apps/p13_demo/03_widgets.py
# ============================================================
# Demo 03 — Widget Input
# Jalankan: streamlit run streamlit_apps/p13_demo/03_widgets.py
# ============================================================
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Widget Input", page_icon="🎛️", layout="wide")
st.title("🎛️ Widget Input — Demo Interaktif")

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")
    tema = st.radio("Tampilan", ["Terang", "Gelap"], horizontal=True)
    st.divider()
    st.caption("Semua widget di bawah bisa kamu coba!")

st.divider()

# ── Row 1: Teks & Angka ──────────────────────────────────
st.header("1. Input Teks & Angka")
col1, col2, col3 = st.columns(3)

with col1:
    nama = st.text_input("Nama proyek", placeholder="misal: Prediksi Churn")
    st.write(f"Kamu ketik: `{nama}`")

with col2:
    n_data = st.number_input("Jumlah data", min_value=10, max_value=100000, value=1000, step=100)
    st.write(f"Jumlah: **{n_data:,}** sampel")

with col3:
    deskripsi = st.text_area("Deskripsi model", placeholder="Tulis deskripsi singkat...", height=100)

st.divider()

# ── Row 2: Slider ─────────────────────────────────────────
st.header("2. Slider")
col1, col2 = st.columns(2)

with col1:
    k = st.slider("Jumlah cluster K (K-Means)", min_value=2, max_value=10, value=3)
    st.info(f"K yang dipilih: **{k}** cluster")

with col2:
    rentang = st.slider("Rentang usia responden", min_value=17, max_value=65, value=(20, 35))
    st.info(f"Usia: **{rentang[0]}** – **{rentang[1]}** tahun")

st.divider()

# ── Row 3: Pilihan ────────────────────────────────────────
st.header("3. Pilihan (Selectbox, Multiselect, Radio)")
col1, col2, col3 = st.columns(3)

with col1:
    model = st.selectbox("Pilih Model ML", ["Random Forest", "SVM", "Logistic Regression", "XGBoost"])
    st.write(f"Model dipilih: **{model}**")

with col2:
    fitur = st.multiselect("Pilih Fitur", ["Usia", "Pendapatan", "Pendidikan", "Lokasi", "Pekerjaan"],
                           default=["Usia", "Pendapatan"])
    st.write(f"Fitur terpilih: {len(fitur)} fitur")

with col3:
    metrik = st.radio("Metrik Evaluasi", ["Accuracy", "F1-Score", "AUC-ROC"], index=1)
    st.write(f"Metrik: **{metrik}**")

st.divider()

# ── Row 4: Toggle, Button, File Upload ───────────────────
st.header("4. Toggle, Button & Upload")
col1, col2 = st.columns(2)

with col1:
    debug = st.toggle("Mode Debug")
    if debug:
        st.warning("⚠️ Mode debug aktif — semua output ditampilkan")

    normalize = st.checkbox("Normalisasi fitur sebelum training?", value=True)
    if normalize:
        st.info("StandardScaler akan digunakan")

with col2:
    uploaded = st.file_uploader("Upload Dataset (.csv)", type=["csv"])
    if uploaded:
        df = pd.read_csv(uploaded)
        st.success(f"✅ File dimuat: {df.shape[0]:,} baris × {df.shape[1]} kolom")
        st.dataframe(df.head(3))
    else:
        st.info("Belum ada file yang diupload")

st.divider()

# ── Demo: Nilai semua widget ──────────────────────────────
with st.expander("🔍 Lihat nilai semua widget (untuk debugging)"):
    st.json({
        "nama": nama,
        "n_data": n_data,
        "k": k,
        "rentang_usia": list(rentang),
        "model": model,
        "fitur": fitur,
        "metrik": metrik,
        "debug": debug,
        "normalize": normalize,
    })

Overwriting ../streamlit_apps/p13_demo/03_widgets.py


---

## BAGIAN 4: Layout

Layout membantu mengatur **struktur visual** app agar konten terorganisir dan mudah dibaca.

### Komponen Layout Utama

```
st.columns([1, 2, 1])              st.tabs(["Tab A", "Tab B"])
┌──────┬────────────┬──────┐        ┌────────────────────────┐
│ col1 │    col2    │ col3 │        │ [Tab A] [Tab B]        │
│ (1x) │    (2x)    │ (1x) │        ├────────────────────────┤
└──────┴────────────┴──────┘        │ Konten Tab A           │
                                    └────────────────────────┘

st.sidebar                          st.expander("Lihat detail")
┌────────┬───────────────────┐      ┌────────────────────────┐
│Sidebar │                   │      │ ▶ Lihat detail         │
│        │   Konten Utama    │  →   ├────────────────────────┤
│ Widget │                   │      │ Detail tersembunyi      │
└────────┴───────────────────┘      └────────────────────────┘
```

### Tips Proporsi Kolom

```python
# Kolom sama lebar
col1, col2, col3 = st.columns(3)

# Kolom dengan proporsi berbeda (angka = rasio lebar)
col_kiri, col_tengah, col_kanan = st.columns([1, 3, 1])
# → kiri sempit, tengah lebar (3x), kanan sempit

# Pakai kolom langsung sebagai context manager
with col1:
    st.metric("MAE", "12.3")
with col2:
    st.metric("RMSE", "18.7")
```

### st.container() — Grup Elemen

```python
# Berguna untuk update konten secara dinamis
placeholder = st.empty()   # bisa diganti kontennya nanti
placeholder.write("Loading...")
# ... proses ...
placeholder.success("Selesai!")  # replace konten sebelumnya
```

---
▶️ **Setelah menjalankan sel berikut**, ketik `launch('04_layout.py')`

In [7]:
%%writefile ../streamlit_apps/p13_demo/04_layout.py
# ============================================================
# Demo 04 — Layout (columns, sidebar, tabs, expander)
# Jalankan: streamlit run streamlit_apps/p13_demo/04_layout.py
# ============================================================
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Layout Demo", page_icon="📐", layout="wide")

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.title("📐 Layout Demo")
    st.divider()
    layout_demo = st.radio(
        "Pilih Demo Layout",
        ["Columns", "Tabs", "Expander", "Sidebar"]
    )
    st.divider()
    st.caption("Sidebar cocok untuk: filter, pengaturan, navigasi")

st.title("📐 Komponen Layout Streamlit")
st.divider()

# ── Demo Columns ───────────────────────────────────────────
st.header("1. st.columns() — Tata Letak Horizontal")

# Proporsi sama
st.subheader("Sama lebar (columns(3))")
col1, col2, col3 = st.columns(3)
col1.metric("MAE",  "12.34", "-1.2")
col2.metric("RMSE", "18.56", "-2.1")
col3.metric("R²",   "0.894", "+0.03")

st.subheader("Proporsi berbeda (columns([1, 2, 1]))")
kiri, tengah, kanan = st.columns([1, 2, 1])
with kiri:
    st.info("Kolom kiri (sempit) — cocok untuk label atau ikon")
with tengah:
    st.success("Kolom tengah (lebar) — cocok untuk chart atau tabel utama")
with kanan:
    st.warning("Kolom kanan (sempit)")

st.divider()

# ── Demo Tabs ─────────────────────────────────────────────
st.header("2. st.tabs() — Konten Berlapis")
tab1, tab2, tab3 = st.tabs(["📊 Data", "📈 Visualisasi", "⚙️ Model Info"])

with tab1:
    np.random.seed(42)
    df = pd.DataFrame({
        'Fitur A': np.random.randn(20),
        'Fitur B': np.random.randn(20),
        'Label':   np.random.choice(['Kelas 0', 'Kelas 1'], 20)
    })
    st.dataframe(df, use_container_width=True)

with tab2:
    st.write("Di sini bisa ditaruh chart, plot, atau visualisasi interaktif")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(df['Fitur A'], df['Fitur B'],
               c=['#3498db' if l == 'Kelas 0' else '#e74c3c' for l in df['Label']], alpha=0.7, s=60)
    ax.set_xlabel("Fitur A")
    ax.set_ylabel("Fitur B")
    ax.set_title("Scatter Plot Fitur A vs Fitur B")
    st.pyplot(fig)

with tab3:
    st.json({
        "model": "Random Forest",
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
        "train_accuracy": 0.962,
        "test_accuracy": 0.914
    })

st.divider()

# ── Demo Expander ─────────────────────────────────────────
st.header("3. st.expander() — Konten Dapat Disembunyikan")

with st.expander("📋 Klik untuk melihat detail preprocessing"):
    st.markdown("""
    **Langkah Preprocessing:**
    1. Drop kolom yang memiliki missing value > 30%
    2. Imputasi median untuk fitur numerik
    3. One-hot encoding untuk fitur kategorikal
    4. StandardScaler untuk normalisasi
    5. Train/test split 80:20
    """)

with st.expander("💡 Tips penggunaan model ini"):
    st.info("Model ini dilatih pada data 2020–2023. Performa mungkin turun untuk data di luar rentang tersebut.")

Overwriting ../streamlit_apps/p13_demo/04_layout.py


---

## BAGIAN 5: Visualisasi & Tampilan Data

Streamlit mendukung berbagai library visualisasi Python. Yang paling umum digunakan:

| Fungsi | Library | Keunggulan |
|--------|---------|-----------|
| `st.pyplot(fig)` | Matplotlib / Seaborn | Familiar, banyak contoh |
| `st.plotly_chart(fig)` | Plotly | Interaktif (hover, zoom, klik) |
| `st.altair_chart(chart)` | Altair | Deklaratif, grammar of graphics |
| `st.dataframe(df)` | Pandas | Tabel interaktif bawaan |
| `st.table(df)` | Pandas | Tabel statis |

### Cara Pakai st.pyplot()

```python
import matplotlib.pyplot as plt
import streamlit as st

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([1, 2, 3], [10, 20, 15])
ax.set_title("Contoh Plot")

st.pyplot(fig)          # tampilkan di app
plt.close(fig)          # tutup figure untuk hemat memori
```

### st.dataframe() — Tabel Interaktif

```python
st.dataframe(
    df,
    use_container_width=True,  # lebar penuh
    height=300,                 # tinggi maksimum
    column_config={             # kustomisasi kolom
        "harga": st.column_config.NumberColumn("Harga", format="Rp %.0f"),
        "akurasi": st.column_config.ProgressColumn("Akurasi", min_value=0, max_value=1),
    }
)
```

### st.spinner() & st.progress() — Loading Feedback

```python
with st.spinner("Melatih model... mohon tunggu"):
    model.fit(X_train, y_train)    # operasi berat
    # spinner muncul selama blok ini berjalan
st.success("Selesai!")
```

---

## BAGIAN 6: Demo App Lengkap — Iris EDA Explorer 🌸

Kita gabungkan semua konsep (teks, widget, layout, visualisasi) ke dalam satu app yang utuh.

### Fitur App yang Akan Dibuat

```
┌─────────────────────────────────────────────────────────────────┐
│  🌸 Iris EDA Explorer                                           │
│  ─────────────────────────────────────────────────────────────  │
│  SIDEBAR              │  KONTEN UTAMA                           │
│  ─────────────────    │  ┌──────────────────────────────────┐  │
│  Filter Spesies:      │  │  📊 Iris EDA Explorer            │  │
│  ☑ setosa             │  │  Menampilkan 120 dari 150 sampel  │  │
│  ☑ versicolor         │  │                                  │  │
│  ☐ virginica          │  │  [150] [4]  [3]  [0]            │  │
│                       │  │  Total Fitur Spesies Missing     │  │
│  Sumbu X: [petal_len] │  │  ─────────────────────────────  │  │
│  Sumbu Y: [petal_wid] │  │  [Scatter Plot][Distribusi][🤖] │  │
│                       │  │                                  │  │
│  ☑ Tampilkan Tabel    │  │  (chart di sini)                 │  │
│                       │  └──────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

Jalankan dengan: `streamlit run streamlit_apps/p13_demo/iris_explorer.py`

---
▶️ **Setelah menjalankan sel berikut**, ketik `launch('iris_explorer.py')`

In [8]:
%%writefile ../streamlit_apps/p13_demo/iris_explorer.py
# ============================================================
# Demo Lengkap — Iris EDA Explorer
# Menggabungkan: sidebar, widgets, columns, tabs, charts
# Jalankan: streamlit run streamlit_apps/p13_demo/iris_explorer.py
# ============================================================
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ── Konfigurasi ───────────────────────────────────────────
st.set_page_config(
    page_title="Iris EDA Explorer",
    page_icon="🌸",
    layout="wide",
    initial_sidebar_state="expanded",
)

WARNA = {'setosa': '#3498db', 'versicolor': '#e74c3c', 'virginica': '#2ecc71'}

# ── Load Data ─────────────────────────────────────────────
@st.cache_data
def load_data():
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    df['species'] = [iris.target_names[i] for i in iris.target]
    return df, iris

df, iris_raw = load_data()

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.title("🌸 Iris Explorer")
    st.caption("Dataset klasik 150 sampel bunga Iris")
    st.divider()

    spesies_pilihan = st.multiselect(
        "Filter Spesies",
        options=sorted(df['species'].unique()),
        default=sorted(df['species'].unique()),
    )

    st.divider()

    col_fitur = [c for c in df.columns if c != 'species']
    fitur_x = st.selectbox("Sumbu X (Scatter Plot)", col_fitur, index=2)
    fitur_y = st.selectbox("Sumbu Y (Scatter Plot)", col_fitur, index=3)

    st.divider()

    tampilkan_tabel = st.checkbox("Tampilkan Tabel Data", value=False)
    tampilkan_stats = st.checkbox("Tampilkan Statistik Deskriptif", value=True)

# Filter data berdasarkan sidebar
if not spesies_pilihan:
    st.error("⚠️ Pilih minimal satu spesies di sidebar!")
    st.stop()

df_f = df[df['species'].isin(spesies_pilihan)]

# ── Header ────────────────────────────────────────────────
st.title("🌸 Iris Dataset — EDA Explorer")
st.markdown(f"Menampilkan **{len(df_f):,}** dari {len(df):,} sampel · "
            f"Filter: {', '.join(spesies_pilihan)}")
st.divider()

# ── Metrics row ───────────────────────────────────────────
c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Sampel",  len(df_f),                  f"{len(df_f)-150}")
c2.metric("Fitur",         len(col_fitur))
c3.metric("Spesies",       df_f['species'].nunique())
c4.metric("Missing Values",df_f.isnull().sum().sum())

st.divider()

# ── Tabs ──────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["📊 Scatter Plot", "📈 Distribusi", "🤖 Demo Model ML"])

# Tab 1: Scatter Plot
with tab1:
    col_plot, col_info = st.columns([3, 1])
    with col_plot:
        fig, ax = plt.subplots(figsize=(8, 5))
        for sp, grp in df_f.groupby('species'):
            ax.scatter(grp[fitur_x], grp[fitur_y],
                       label=sp.capitalize(), color=WARNA[sp], alpha=0.75, s=60, edgecolors='white', linewidth=0.5)
        ax.set_xlabel(fitur_x.replace('_', ' ').title(), fontsize=12)
        ax.set_ylabel(fitur_y.replace('_', ' ').title(), fontsize=12)
        ax.set_title(f'{fitur_x} vs {fitur_y}', fontsize=13)
        ax.legend()
        sns.despine()
        st.pyplot(fig)
        plt.close(fig)
    with col_info:
        st.subheader("Korelasi")
        corr = df_f[[fitur_x, fitur_y]].corr().iloc[0, 1]
        st.metric("r Pearson", f"{corr:.3f}")
        if abs(corr) > 0.7:
            st.success("Korelasi kuat")
        elif abs(corr) > 0.4:
            st.warning("Korelasi sedang")
        else:
            st.info("Korelasi lemah")

        if tampilkan_stats:
            st.subheader("Statistik")
            st.dataframe(df_f[[fitur_x, fitur_y]].describe().round(2), use_container_width=True)

# Tab 2: Distribusi
with tab2:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, fitur in zip(axes.flatten(), col_fitur):
        for sp, grp in df_f.groupby('species'):
            ax.hist(grp[fitur], alpha=0.55, label=sp.capitalize(),
                    bins=15, color=WARNA[sp], edgecolor='white')
        ax.set_title(fitur.replace('_', ' ').title(), fontsize=11)
        ax.set_xlabel('cm')
        ax.legend(fontsize=8)
        sns.despine(ax=ax)
    plt.suptitle('Distribusi Fitur per Spesies', fontsize=13)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

# Tab 3: Demo Model
with tab3:
    st.subheader("🤖 Klasifikasi Iris dengan Random Forest")
    st.markdown("Coba ubah parameter di bawah dan klik **Latih Model** untuk melihat pengaruhnya.")

    col_p1, col_p2, col_p3 = st.columns(3)
    with col_p1:
        n_trees  = st.slider("n_estimators (jumlah pohon)", 10, 300, 100, 10)
    with col_p2:
        max_dep  = st.select_slider("max_depth", options=[None, 3, 5, 10, 20], value=None)
    with col_p3:
        ts       = st.slider("Test size (%)", 10, 40, 20, 5) / 100

    if st.button("🚀 Latih Model", type="primary", use_container_width=True):
        with st.spinner("Melatih Random Forest..."):
            X = df[col_fitur].values
            y = iris_raw.target
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
            clf = RandomForestClassifier(n_estimators=n_trees, max_depth=max_dep, random_state=42)
            clf.fit(X_tr, y_tr)
            acc_tr = accuracy_score(y_tr, clf.predict(X_tr))
            acc_te = accuracy_score(y_te, clf.predict(X_te))

        st.success("✅ Model selesai dilatih!")
        m1, m2, m3 = st.columns(3)
        m1.metric("Akurasi Train", f"{acc_tr:.2%}")
        m2.metric("Akurasi Test",  f"{acc_te:.2%}", f"{acc_te - acc_tr:.2%}")
        m3.metric("Jumlah Test",   len(y_te))

        # Feature importance
        fi = pd.Series(clf.feature_importances_, index=col_fitur).sort_values()
        fig, ax = plt.subplots(figsize=(6, 3))
        fi.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
        ax.set_title('Feature Importance', fontsize=11)
        ax.set_xlabel('Importance')
        sns.despine(ax=ax)
        plt.tight_layout()
        st.pyplot(fig)
        plt.close(fig)

        with st.expander("📋 Classification Report (Test Set)"):
            cr = classification_report(y_te, clf.predict(X_te), target_names=iris_raw.target_names)
            st.text(cr)

# ── Tabel Data ────────────────────────────────────────────
if tampilkan_tabel:
    st.divider()
    st.subheader("📋 Data yang Difilter")
    st.dataframe(df_f.reset_index(drop=True), use_container_width=True, height=300)

Overwriting ../streamlit_apps/p13_demo/iris_explorer.py


---

## ✏️ Latihan Mandiri

Kerjakan soal berikut. Setiap latihan adalah file `.py` baru yang bisa langsung dijalankan dengan `streamlit run`.

> **Tips:** Mulai dari copy-paste kode demo di atas, lalu modifikasi sesuai instruksi.

> **Solusi lengkap** sudah disediakan di setiap sel latihan — jalankan langsung dengan `launch('latihan_X_....py')`.

In [ ]:
%%writefile ../streamlit_apps/p13_demo/latihan_1_kalkulator.py
# LATIHAN 1: Kalkulator BMI Sederhana (SOLUSI LENGKAP)
# ============================================================
import streamlit as st

st.set_page_config(page_title="Kalkulator BMI", page_icon="⚖️")
st.title("⚖️ Kalkulator BMI")
st.markdown("Masukkan berat dan tinggi badan untuk menghitung BMI")

col1, col2 = st.columns(2)
with col1:
    berat = st.number_input("Berat Badan (kg)", 20.0, 300.0, 65.0, 0.5)
with col2:
    tinggi = st.number_input("Tinggi Badan (cm)", 50.0, 250.0, 165.0, 0.5)

if tinggi > 0:
    bmi = berat / ((tinggi / 100) ** 2)
    st.subheader("Hasil")
    st.metric("BMI Anda", f"{bmi:.1f}")

    if bmi < 18.5:
        st.warning("Kategori: Kurus (Underweight)")
    elif bmi < 25.0:
        st.success("Kategori: Normal (Ideal)")
    elif bmi < 30.0:
        st.warning("Kategori: Gemuk (Overweight)")
    else:
        st.error("Kategori: Obesitas")

    st.progress(min(bmi / 40, 1.0))
else:
    st.info("Masukkan tinggi dan berat badan untuk menghitung BMI")

# ▶️ Jalankan: launch('latihan_1_kalkulator.py')


In [ ]:
%%writefile ../streamlit_apps/p13_demo/latihan_2_upload_eda.py
# LATIHAN 2: EDA Tool untuk Dataset Tim (SOLUSI LENGKAP)
# ============================================================
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="EDA Tool", page_icon="🔍", layout="wide")
st.title("🔍 EDA Tool — Upload Dataset Timmu")

uploaded = st.file_uploader("Upload file CSV", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)

    c1, c2, c3 = st.columns(3)
    c1.metric("Jumlah Baris", df.shape[0])
    c2.metric("Jumlah Kolom", df.shape[1])
    c3.metric("Missing Values", df.isnull().sum().sum())

    tab1, tab2, tab3 = st.tabs(["📋 Preview", "📊 Statistik", "📈 Visualisasi"])

    with tab1:
        st.dataframe(df.head(10), use_container_width=True)

    with tab2:
        st.dataframe(df.describe().round(2), use_container_width=True)

    with tab3:
        num_cols = df.select_dtypes(include="number").columns.tolist()
        if num_cols:
            col = st.selectbox("Pilih kolom numerik", num_cols)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(df[col].dropna(), bins=30, color="#3498db", edgecolor="white")
            ax.set_title(f"Distribusi {col}")
            ax.set_xlabel(col)
            ax.set_ylabel("Frekuensi")
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("Tidak ada kolom numerik untuk divisualisasikan")

    with st.expander("🔗 Correlation Heatmap"):
        if len(num_cols) > 1:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu", ax=ax)
            ax.set_title("Heatmap Korelasi")
            st.pyplot(fig)
            plt.close(fig)
        else:
            st.info("Butuh minimal 2 kolom numerik untuk heatmap korelasi")
else:
    st.info("Silakan upload file CSV untuk memulai eksplorasi")

# ▶️ Jalankan: launch('latihan_2_upload_eda.py')


In [ ]:
%%writefile ../streamlit_apps/p13_demo/latihan_3_ml_demo.py
# LATIHAN 3 (Tantangan): Demo Model ML Tim (SOLUSI LENGKAP)
# ============================================================
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

st.set_page_config(page_title="Demo ML Tim", page_icon="🤖", layout="wide")
st.title("🤖 Demo Model ML — Upload & Latih")

uploaded = st.file_uploader("Upload dataset CSV", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)
    st.success(f"Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
    st.dataframe(df.head(), use_container_width=True)

    with st.sidebar:
        st.header("⚙️ Pengaturan")
        target = st.selectbox("Kolom Target", df.columns)
        cols = [c for c in df.columns if c != target]
        X_cols = st.multiselect("Pilih Fitur", cols,
                                default=cols[:min(3, len(cols))])
        model_name = st.selectbox("Pilih Model",
            ["Logistic Regression", "Decision Tree", "Random Forest"])

        if st.button("🚀 Latih Model", type="primary", use_container_width=True):
            if len(X_cols) == 0:
                st.error("Pilih minimal 1 fitur!")
                st.stop()

            with st.spinner("Melatih model..."):
                X = df[X_cols].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
                y = df[target]
                if y.dtype == 'object':
                    y = LabelEncoder().fit_transform(y.values.ravel() if hasattr(y, 'values') else y)

                X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

                scaler = StandardScaler()
                X_tr_s = scaler.fit_transform(X_tr)
                X_te_s = scaler.transform(X_te)

                models = {
                    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
                    "Decision Tree": DecisionTreeClassifier(random_state=42),
                    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
                }
                model = models[model_name]
                model.fit(X_tr_s, y_tr)

                acc_tr = accuracy_score(y_tr, model.predict(X_tr_s))
                acc_te = accuracy_score(y_te, model.predict(X_te_s))

            c1, c2 = st.columns(2)
            c1.metric("Akurasi Train", f"{acc_tr:.2%}")
            c2.metric("Akurasi Test", f"{acc_te:.2%}")

            st.subheader("Classification Report")
            cr = classification_report(y_te, model.predict(X_te_s), zero_division=0)
            st.text(cr)

            if hasattr(model, "feature_importances_"):
                fi = pd.Series(model.feature_importances_, index=X_cols).sort_values()
                fig, ax = plt.subplots(figsize=(8, 4))
                fi.plot(kind="barh", ax=ax, color="#3498db", edgecolor="white")
                ax.set_title("Feature Importance")
                ax.set_xlabel("Importance")
                st.pyplot(fig)
                plt.close(fig)
else:
    st.info("Upload file CSV untuk memulai training model")

# ▶️ Jalankan: launch('latihan_3_ml_demo.py')


---

## 📝 Ringkasan Pertemuan 13

```
STREAMLIT — ALUR DASAR MEMBANGUN APP

Konsep & Data  →  Layout  →  Widget  →  Logika  →  Display
─────────────────────────────────────────────────────────────
import streamlit as st

st.set_page_config(...)          # selalu pertama
                                 
with st.sidebar:                 # sidebar: navigasi & filter
    pilihan = st.selectbox(...)
    nilai   = st.slider(...)

col1, col2 = st.columns([2, 1])  # layout horizontal
with col1:
    fig, ax = plt.subplots()
    ax.plot(...)
    st.pyplot(fig)               # tampilkan chart

with col2:
    st.metric("Akurasi", "94%")  # KPI

with st.tabs(["A", "B"]):        # konten berlapis
    ...
```

### Recap: Komponen yang Sudah Dipelajari

| Kategori | Komponen |
|----------|---------|
| **Teks** | `title`, `header`, `write`, `markdown`, `code`, `metric` |
| **Status** | `success`, `error`, `warning`, `info` |
| **Widget** | `text_input`, `number_input`, `slider`, `selectbox`, `multiselect`, `radio`, `checkbox`, `button`, `file_uploader` |
| **Layout** | `columns`, `sidebar`, `tabs`, `expander`, `divider` |
| **Visualisasi** | `pyplot`, `dataframe`, `spinner`, `stop` |

---

**Pertemuan berikutnya (P14):** Caching, Session State, Multi-page Apps, Full ML App California Housing, dan **Deploy ke Streamlit Cloud** 🚀

---

## 🚀 Quick Launch — Jalankan Semua Demo dari Sini

Gunakan tombol ▶️ di sel berikut untuk meluncurkan app pilihanmu langsung di browser.

In [ ]:
# Pilih demo yang ingin dijalankan, lalu ▶️ Run
pilihan = "iris_explorer.py"  # ← ganti dengan nama file yang diinginkan

# Daftar semua demo yang tersedia:
# launch('01_hello.py')
# launch('02_teks_display.py')
# launch('03_widgets.py')
# launch('04_layout.py')
launch(pilihan)  # ← jalankan baris ini

# Latihan (sudah lengkap dengan solusi):
# launch('latihan_1_kalkulator.py')
# launch('latihan_2_upload_eda.py')
# launch('latihan_3_ml_demo.py')

---
> **💡 Tips:** Kamu juga bisa menjalankan manual dari terminal:
> ```
> streamlit run streamlit_apps/p13_demo/nama_file.py
> ```
> Tekan **Ctrl+C** di terminal untuk menghentikan app.